# Los datos de la web app

Este pipeline no calcula nada nuevo: toma las tablas del 1_ y las escribe en `data/clean` en el
formato que consume el frontend Todo va en JSON por columnas y con las
coordenadas ya proyectadas a EPSG:5346 en hectómetros enteros, así el navegador mide distancias sin
proyectar nada. La gente que ya tiene tren de pasajeros no va punto por punto, porque la red de hoy
no se puede achicar y esa gente no cambia de categoría (para el checkpoint de "reconectar")

## Los archivos

Cada archivo es un objeto de listas paralelas: la posición k de cada lista es el elemento k. Las
coordenadas y distancias están en hectómetros enteros en EPSG:5346, y las líneas van como `[x0, y0,
dx1, dy1, ...]`: el primer punto y después cuánto se mueve cada uno.

- `vias.json` — una entrada por parte de vía. `categoria` (0 levantada, 1 inactiva, 2 carga,
  3 pasajeros), `alta`, `fin_pasajeros` y `baja` (años; `fin_pasajeros` es `null` en la vía que
  todavía lleva pasajeros y `baja` en la vía activa), `km`, `estaciones` (índices en `estaciones.json` a
  menos de 500 m) y `xy`.
- `estaciones.json` — una entrada por estación que cuenta. `x`, `y` y `nivel` (1 perdió el tren,
  2 sobre vía de carga, 3 sobre vía con pasajeros).
- `poblacion.json` — una entrada por punto de gente de 2022 sin tren de pasajeros con la red de
  hoy. `x`, `y`, `pob`, `dep` (índice en `regiones.departamentos`) y `nivel` (0 nunca tuvo tren,
  1 perdió el tren, 2 sólo carga).
- `regiones.json` — `departamentos`: `codigo`, `x` e `y`, el centro de su gente sin tren;
  `provincias`: `codigo` y `nombre`.
- `mapa_base.json` — `anillos`: el contorno de cada polígono de provincia, codificado como `xy`.
- `serie.json` — una entrada por año. `anio`, `cerradas` (`null` fuera de 1940–2025),
  `km_pasajeros`, `km_carga`, `km_inactiva` y `km_levantada`, la vía construida hasta ese año en
  cada categoría.
- `censos.json` — una entrada por censo, de 1991 a 2022. `censo` y `personas`, la gente con tren de
  pasajeros, sólo de carga, que perdió el tren y que nunca tuvo, con la red de ese año.
- `censos_departamentos.json` — la misma gente, por departamento de 2022. `departamentos`: `codigo`,
  `x` e `y`, el centro de su gente en 2022; `personas`: para cada censo y cada una de las
  categorías de `censos.json`, la gente de cada departamento.
- `meta.json` — `alcance` y `celda_grilla` en hectómetros, y la `atribucion` que piden las
  licencias.

La categoría de una parte en un año sale de `vias.json` con la misma regla del 1_: no existe antes
de `alta`, lleva pasajeros hasta `fin_pasajeros`, es de carga hasta `baja` y después queda en su
`categoria` de hoy, inactiva o levantada.

## Datos

Todo del 1_, en `data/pipeline`: `vias.parquet`, `estaciones.parquet`, `estaciones_vias.csv`,
`radios.parquet`, `provincias.parquet` y `provincias_nombres.csv`.

In [8]:
import json, gzip
import numpy as np, pandas as pd, geopandas as gpd
from pathlib import Path

PIPELINE = Path('data/pipeline')
LIMPIO = Path('data/clean')
LIMPIO.mkdir(parents=True, exist_ok=True)

HM = 100                       # metros por unidad: todo se exporta en hectómetros enteros
ALCANCE = 10_000               # metros, el UMBRAL_KM del 1_
CELDA_POBLACION = 2_000        # metros de lado de la grilla en que se junta la gente
CELDA_GRILLA = 25_000          # metros de lado de la grilla con que el frontend localiza cada recálculo
ANIO_MIN, ANIO_MAX = 1857, 2025
CIERRES_DESDE = 1940           # antes, Wikidata casi no tiene fechas de cierre

CATEGORIAS = ['levantada', 'inactiva', 'carga', 'pasajeros']   # el código es la posición
PASAJEROS, CARGA = 'con tren de pasajeros', 'sólo tren de carga'
CERRADA, NUNCA = 'perdió el tren', 'nunca tuvo tren'
NIVELES = [NUNCA, CERRADA, CARGA, PASAJEROS]                   # el código es la posición
TIPOS = [PASAJEROS, CARGA, CERRADA, NUNCA]

vias = gpd.read_parquet(PIPELINE / 'vias.parquet')
estaciones = pd.read_parquet(PIPELINE / 'estaciones.parquet')
estaciones_vias = pd.read_csv(PIPELINE / 'estaciones_vias.csv')
radios = pd.read_parquet(PIPELINE / 'radios.parquet')

def hm(valores):
    return np.round(np.asarray(valores, dtype=float) / HM).astype(int)

def en_diferencias(coords):
    # Como TopoJSON: el primer punto entero y después sólo cuánto se mueve, que comprime mucho mejor.
    xy = hm(coords)
    return np.r_[xy[0], np.diff(xy, axis=0).ravel()].tolist()

def entero_o_nada(valores):
    lista = []
    for v in valores:
        lista.append(None if np.isnan(v) else int(v))
    return lista

def guardar(nombre, datos):
    texto = json.dumps(datos, separators=(',', ':'), ensure_ascii=False).encode()
    (LIMPIO / nombre).write_bytes(texto)
    print(f'{nombre:26s} {len(texto) / 1e3:6.0f} KB · gzip {len(gzip.compress(texto, 9)) / 1e3:4.0f} KB')

## Las estaciones

Sólo las que cuentan, con su nivel de hoy: la mejor categoría de las partes que tienen a menos de
500 m, igual que en el 1_.

In [9]:
cuentan = estaciones[estaciones.cuenta]
fila_a_indice = pd.Series(np.arange(len(cuentan)), index=cuentan.index)
nivel_parte = vias.categoria.map({'pasajeros': 3, 'carga': 2}).fillna(1).values
nivel = np.ones(len(cuentan), int)
np.maximum.at(nivel, fila_a_indice[estaciones_vias.estacion].values, nivel_parte[estaciones_vias.parte])
guardar('estaciones.json', {'x': hm(cuentan.x).tolist(), 'y': hm(cuentan.y).tolist(), 'nivel': nivel.tolist()})
print(pd.Series(nivel).value_counts().sort_index().to_string())

estaciones.json                43 KB · gzip   16 KB
1    1359
2    1170
3     532


## Las vías

Cada parte lleva su categoría de hoy, sus años para la línea de tiempo, su largo, las estaciones que
tiene a menos de 500 m y su línea.

In [10]:
est_de_parte = [[] for _ in range(len(vias))]
for e, parte in sorted(zip(fila_a_indice[estaciones_vias.estacion].values, estaciones_vias.parte)):
    est_de_parte[parte].append(int(e))

fin_pasajeros = np.where(vias.categoria == 'pasajeros', np.nan, vias.baja)
baja = np.where(vias.activo, np.nan, vias.baja)   # la línea de tiempo no mira la baja de la vía activa
guardar('vias.json', {'categoria': [CATEGORIAS.index(c) for c in vias.categoria],
                      'alta': vias.alta.astype(int).tolist(), 'fin_pasajeros': entero_o_nada(fin_pasajeros),
                      'baja': entero_o_nada(baja), 'km': vias.km.round(2).tolist(),
                      'estaciones': est_de_parte,
                      'xy': [en_diferencias(g.coords) for g in vias.geometry]})
print(f'{len(vias)} partes · {sum(1 for l in est_de_parte if l)} partes con alguna estación')

vias.json                     416 KB · gzip  107 KB
7932 partes · 4069 partes con alguna estación


## La gente sin tren de pasajeros

La gente de 2022 sin tren de pasajeros con la red de hoy, que es la que Reconectar puede sumar. Los
radios se juntan en una grilla de 2 km, separados por departamento y por categoría, en el centro de
su gente: achica los datos a la mitad y casi no mueve nada, porque la celda es mucho más chica que
los 10 km de alcance.

In [11]:
r22 = radios[radios.censo == 2022].copy()
r22['px'] = r22.x * r22.pob
r22['py'] = r22.y * r22.pob
r22['gx'] = np.floor(r22.x / CELDA_POBLACION)
r22['gy'] = np.floor(r22.y / CELDA_POBLACION)
sin_tren = r22[r22.acceso_hoy != PASAJEROS].copy()
sin_tren['nivel'] = sin_tren.acceso_hoy.map({n: k for k, n in enumerate(NIVELES)})
puntos = sin_tren.groupby(['gx', 'gy', 'dep', 'nivel']).agg(pob=('pob', 'sum'), px=('px', 'sum'),
                                                          py=('py', 'sum')).reset_index()
puntos = puntos[puntos.pob > 0]
deps = sorted(puntos.dep.unique())
indice_dep = {d: k for k, d in enumerate(deps)}
guardar('poblacion.json', {'x': hm(puntos.px / puntos.pob).tolist(), 'y': hm(puntos.py / puntos.pob).tolist(),
                           'pob': puntos.pob.astype(int).tolist(), 'dep': puntos.dep.map(indice_dep).tolist(),
                           'nivel': puntos.nivel.tolist()})
print(f'{len(sin_tren)} radios sin tren de pasajeros, {sin_tren.pob.sum() / 1e6:.2f} M personas · '
      f'{len(puntos)} puntos · {len(deps)} departamentos')

poblacion.json                379 KB · gzip  117 KB
38133 radios sin tren de pasajeros, 22.04 M personas · 17529 puntos · 490 departamentos


## Departamentos y provincias

Cada departamento con gente sin tren lleva el centro de esa gente, que es de donde arranca su
círculo en el Dorling. Cada provincia lleva su nombre, que el frontend busca por los dos primeros
dígitos del código de departamento.

In [12]:
nombres = pd.read_csv(PIPELINE / 'provincias_nombres.csv', dtype={'cod_prov': str})
provincias = nombres.cod_prov.tolist()
centro = sin_tren.groupby('dep').agg(pob=('pob', 'sum'), px=('px', 'sum'), py=('py', 'sum')).reindex(deps)
guardar('regiones.json', {
    'departamentos': {'codigo': deps, 'x': hm(centro.px / centro.pob).tolist(), 'y': hm(centro.py / centro.pob).tolist()},
    'provincias': {'codigo': provincias, 'nombre': nombres.nombre.tolist()}})

regiones.json                  10 KB · gzip    4 KB


## La gente de cada censo

La gente de cada categoría en los cuatro censos, cada uno con la red de su año, en total y por
departamento de 2022. El centro de cada departamento es el de su gente en 2022; Malvinas no tiene
gente en ningún censo y queda afuera. El frontend la dibuja en el mapa cuando se elige una categoría
de un censo.

In [13]:
censos = sorted(radios.censo.unique())
personas = radios.pivot_table(index='censo', columns='acceso', values='pob', aggfunc='sum').reindex(columns=TIPOS)
guardar('censos.json', {'censo': [int(c) for c in censos],
                        'personas': personas.round().astype(int).values.tolist()})

centro = r22.groupby('dep').agg(pob=('pob', 'sum'), px=('px', 'sum'), py=('py', 'sum'))
centro = centro[centro.pob > 0]
deps_censos = centro.index.tolist()
por_dep = radios.pivot_table(index=['censo', 'dep'], columns='acceso', values='pob', aggfunc='sum', fill_value=0)
personas_dep = []
for censo in censos:
    tabla = por_dep.loc[censo].reindex(index=deps_censos, columns=TIPOS, fill_value=0)
    personas_dep.append([tabla[t].round().astype(int).tolist() for t in TIPOS])
guardar('censos_departamentos.json', {
    'departamentos': {'codigo': deps_censos, 'x': hm(centro.px / centro.pob).tolist(),
                      'y': hm(centro.py / centro.pob).tolist()},
    'personas': personas_dep})

censos.json                     0 KB · gzip    0 KB
censos_departamentos.json      42 KB · gzip   17 KB


## El mapa base, la serie y los parámetros

El contorno de las provincias va con la misma codificación que las vías. La serie suma el `km` de
las partes de cada categoría, año por año, con la regla del 1_, y cuenta las estaciones de Wikidata
que cerraron cada año, tengan o no coordenadas. `meta.json` junta las distancias que tienen que
valer igual en el pipeline y en el frontend, y la atribución que piden las licencias.

In [14]:
anillos = []
for g in gpd.read_parquet(PIPELINE / 'provincias.parquet').geometry:
    partes = g.geoms if g.geom_type == 'MultiPolygon' else [g]
    for parte in partes:
        anillos.append(en_diferencias(parte.exterior.coords))
guardar('mapa_base.json', {'anillos': anillos})

ALTA = vias.alta.values
FIN_PASAJEROS = np.where(vias.categoria == 'pasajeros', np.inf, vias.baja)
FIN_SERVICIO = np.where(vias.categoria.isin(['pasajeros', 'carga']), np.inf, vias.baja)
CATEGORIA_HOY = vias.categoria.values

def categoria_en(anio):
    """La categoría de cada parte en un año, o '' si todavía no estaba construida."""
    construida = ALTA <= anio
    pasajeros = construida & (anio < FIN_PASAJEROS)
    carga = construida & ~pasajeros & (anio < FIN_SERVICIO)
    cerrada = construida & ~pasajeros & ~carga
    return np.select([pasajeros, carga, cerrada], ['pasajeros', 'carga', CATEGORIA_HOY], default='')

anios = np.arange(ANIO_MIN, ANIO_MAX + 1)
km = pd.DataFrame(0, index=anios, columns=CATEGORIAS)
for anio in anios:
    cat = categoria_en(anio)
    for c in CATEGORIAS:
        km.loc[anio, c] = round(vias.km[cat == c].sum())

cierre = estaciones.baja[(estaciones.fuente == 'Wikidata') & estaciones.baja.between(CIERRES_DESDE, ANIO_MAX)]
cerradas = cierre.astype(int).value_counts().reindex(anios)
cerradas[anios >= CIERRES_DESDE] = cerradas[anios >= CIERRES_DESDE].fillna(0)
guardar('serie.json', {'anio': anios.tolist(), 'cerradas': entero_o_nada(cerradas.values),
                       'km_pasajeros': km.pasajeros.tolist(), 'km_carga': km.carga.tolist(),
                       'km_inactiva': km.inactiva.tolist(), 'km_levantada': km.levantada.tolist()})

guardar('meta.json', {
    'alcance': ALCANCE // HM, 'celda_grilla': CELDA_GRILLA // HM,
    'atribucion': ['Vías y estaciones: IDE de la Secretaría de Transporte de la Nación',
                   'Recorridos y vías levantadas: © colaboradores de OpenStreetMap, ODbL',
                   'Población: INDEC, censos 1991, 2001, 2010 y 2022', 'Fechas y estaciones cerradas: Wikidata']})

mapa_base.json                 76 KB · gzip   28 KB
serie.json                      4 KB · gzip    1 KB
meta.json                       0 KB · gzip    0 KB
